[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadErlanggaKurniawan/ets-ml2-kelompok/blob/main/notebooks/eksperimen_michael_ensemble.ipynb)

# Eksperimen Ensemble Tambahan
**Dikerjakan oleh:** Michael Valencio Erasmo Bagas Ganendra · lanjutan dari notebook utama `ETS_ML2_Kesegaran_Ayam.ipynb` (slide 5–7)

Notebook ini menguji **ensemble learning** lebih jauh. Data dan fiturnya sama persis dengan notebook utama, jadi angka
awalnya harus sama dengan slide.

## Cara mengerjakan
1. Jalankan sel dari atas ke bawah (**Shift + Enter**), atau pilih Runtime → Run all.
2. Di sel bertanda **✏️ UBAH DI SINI**, coba ganti angkanya, jalankan lagi, dan perhatikan hasilnya.
3. Di sel bertanda **✍️ TULIS DI SINI**, tulis pengamatanmu dengan kata-katamu sendiri (klik dua kali sel
   itu untuk mengedit).
4. Setiap selesai satu langkah, simpan ke GitHub: **File → Save a copy in GitHub**. Pilih repo
   `MuhammadErlanggaKurniawan/ets-ml2-kelompok`, branch `main`, path `notebooks/eksperimen_michael_ensemble.ipynb`, lalu tulis pesan commit singkat (contoh di tiap langkah).
   Satu kali simpan = satu commit atas nama akunmu.

---
## Langkah 0 — Siapkan data (sama dengan notebook utama)
Sel ini mengunduh data, membuat label, merangkum per 5 menit, dan membuat 19 fitur. Cukup dijalankan, tidak perlu
diubah. Butuh ±10 detik.

In [ ]:
import hashlib, urllib.request, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings("ignore")
SEED = 42
CLASSES = ["Fresh", "Semi-fresh", "Spoiled"]
URL = ("https://raw.githubusercontent.com/talhaanwarch/chicken_assessment_e-nose/"
       "e7e3ca686bce0ef3e5c59f9833c85a3190c36e6e/complete_data.csv")
SHA256 = "7c9920fcfe523af1ee2855d53f13334ddba93db5135d7868b9f63c5cd488d953"
path = Path("data/raw/complete_data.csv")
if not path.exists():
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(URL, path)
assert hashlib.sha256(path.read_bytes()).hexdigest() == SHA256, "Isi file berbeda dari sumber asli"

SENSOR_NAMES = {"s1": "MQ-135", "s2": "MQ-2", "s3": "MQ-3", "s4": "MQ-6", "s5": "MQ-7", "s6": "MQ-9",
                "s8": "Kelembapan"}
raw = pd.read_csv(path).rename(columns={**SENSOR_NAMES, "hour": "jam", "group": "sesi"})
MQ = ["MQ-135", "MQ-2", "MQ-3", "MQ-6", "MQ-7", "MQ-9"]
SENSORS = MQ + ["Kelembapan"]
long_session = raw.groupby("sesi")["jam"].transform("max") > 15
raw["label"] = np.select([raw.jam <= np.where(long_session, 8, 5), raw.jam <= np.where(long_session, 16, 10)],
                         [0, 1], 2)


def make_windows(readings):
    r = readings.copy()
    r["urutan"] = r.groupby(["sesi", "jam"]).cumcount()
    r["n"] = r.groupby(["sesi", "jam"])["jam"].transform("size")
    r["jendela"] = r.urutan * 12 // r.n
    return r.groupby(["sesi", "jam", "jendela"]).agg(**{c: (c, "mean") for c in SENSORS},
                                                    label=("label", "first")).reset_index()


def make_features(win):
    level = np.log1p(win[MQ].clip(lower=0))
    reference = level.where(win.jam == 1).groupby(win.sesi).transform("mean")
    keep = (win.jam >= 2).to_numpy()
    rows = win[keep].reset_index(drop=True)
    level, reference = level[keep].reset_index(drop=True), reference[keep].reset_index(drop=True)
    gas = rows[MQ].clip(lower=0)
    features = pd.concat([level.add_prefix("level_"), gas.div(gas.sum(axis=1) + 1, axis=0).add_prefix("porsi_"),
                          (level - reference).add_prefix("naik_"), rows[["Kelembapan"]]], axis=1)
    return features, rows


features, data = make_features(make_windows(raw))
X, y, groups = features.to_numpy(), data.label.to_numpy(), data.sesi.to_numpy()


def loso(make_model, X_=None):
    # every window is predicted by a model that never saw its session
    X_ = X if X_ is None else X_
    pred, proba = np.empty_like(y), np.zeros((len(y), 3))
    for train, test in LeaveOneGroupOut().split(X_, y, groups):
        model = make_model().fit(X_[train], y[train])
        pred[test], proba[test] = model.predict(X_[test]), model.predict_proba(X_[test])
    return pred, proba


def score(pred):
    return {"akurasi": round(accuracy_score(y, pred), 3), "macro_f1": round(f1_score(y, pred, average="macro"), 3)}


make_rf = lambda: RandomForestClassifier(n_estimators=300, max_depth=6, random_state=SEED, n_jobs=-1)
print(f"Data siap: {len(data)} jendela, {X.shape[1]} fitur, {len(np.unique(groups))} sesi.")

---
## Langkah 1 — Ulangi hasil Random Forest di slide
Hasilnya harus **akurasi 0,776 dan macro-F1 0,782** (slide 6). Kalau beda sedikit di Colab, itu karena versi
library; catat saja.

💾 Pesan commit contoh: `ulangi baseline Random Forest`

In [ ]:
pred_rf, _ = loso(make_rf)
hasil = {"Random Forest (300 pohon)": score(pred_rf)}
print(hasil)

### ✍️ TULIS DI SINI
Apakah angkanya sama dengan slide 6? Kalau beda, berapa bedanya?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 2 — Extra Trees (ensemble bagging yang lebih acak)
Extra Trees mirip Random Forest, tapi titik potong di tiap pohon dipilih **acak**. Kadang ini membuat model lebih
tahan terhadap perbedaan antar sesi.

💾 Pesan commit contoh: `coba Extra Trees`

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

N_POHON = 300      # ✏️ UBAH DI SINI (coba 100 atau 500)
KEDALAMAN = 6      # ✏️ UBAH DI SINI (coba 4 atau 10)

pred_et, _ = loso(lambda: ExtraTreesClassifier(n_estimators=N_POHON, max_depth=KEDALAMAN, random_state=SEED, n_jobs=-1))
hasil[f"Extra Trees ({N_POHON} pohon, kedalaman {KEDALAMAN})"] = score(pred_et)
display(pd.DataFrame(hasil).T)

### ✍️ TULIS DI SINI
Extra Trees lebih baik atau lebih buruk dari Random Forest? Kenapa menurutmu?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 3 — LightGBM (ensemble boosting lain)
LightGBM mirip XGBoost: pohon dibuat berurutan, tiap pohon memperbaiki kesalahan sebelumnya, tapi cara
menumbuhkan pohonnya berbeda (per daun, bukan per tingkat).

💾 Pesan commit contoh: `coba LightGBM`

In [ ]:
from lightgbm import LGBMClassifier

N_POHON_LGBM = 200     # ✏️ UBAH DI SINI
LEARNING_RATE = 0.05   # ✏️ UBAH DI SINI (coba 0.01 atau 0.1)

pred_lgbm, _ = loso(lambda: LGBMClassifier(n_estimators=N_POHON_LGBM, num_leaves=15, learning_rate=LEARNING_RATE,
                                           random_state=SEED, verbose=-1))
hasil[f"LightGBM ({N_POHON_LGBM} pohon, lr {LEARNING_RATE})"] = score(pred_lgbm)
display(pd.DataFrame(hasil).T)

### ✍️ TULIS DI SINI
Bandingkan LightGBM dengan XGBoost di slide 6 (macro-F1 0,765). Mana yang lebih baik?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 4 — Apakah lebih banyak pohon selalu lebih baik?
Random Forest dicoba dengan jumlah pohon berbeda.

💾 Pesan commit contoh: `uji jumlah pohon Random Forest`

In [ ]:
JUMLAH_POHON = [10, 50, 100, 300]   # ✏️ UBAH DI SINI (boleh tambah, misalnya 500)

f1_per_pohon = {}
for n in JUMLAH_POHON:
    pred, _ = loso(lambda: RandomForestClassifier(n_estimators=n, max_depth=6, random_state=SEED, n_jobs=-1))
    f1_per_pohon[n] = score(pred)["macro_f1"]
    print(f"{n:>4} pohon → macro-F1 {f1_per_pohon[n]}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(f1_per_pohon), list(f1_per_pohon.values()), marker="o", color="#0F5257", lw=2)
ax.set(xlabel="jumlah pohon", ylabel="macro-F1 (LOSO)", title="Random Forest: jumlah pohon vs skor")
plt.show()

### ✍️ TULIS DI SINI
Mulai berapa pohon skornya tidak naik lagi? Apa artinya untuk ensemble?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 5 — Model mana yang paling stabil antar sesi?
Akurasi dihitung per sesi untuk tiap model. Model yang bagus bukan cuma rata-ratanya tinggi, tapi juga tidak
terlalu jelek di sesi tertentu.

💾 Pesan commit contoh: `bandingkan akurasi per sesi`

In [ ]:
per_sesi = pd.DataFrame({
    nama: [accuracy_score(y[groups == s], pred[groups == s]) for s in np.unique(groups)]
    for nama, pred in [("Random Forest", pred_rf), ("Extra Trees", pred_et), ("LightGBM", pred_lgbm)]
}, index=np.unique(groups))
per_sesi.index.name = "sesi"
display(per_sesi.round(3))
print("Akurasi terendah tiap model:")
print(per_sesi.min().round(3))

per_sesi.plot.bar(figsize=(10, 4), color=["#0F5257", "#9AA5A6", "#E3A33B"])
plt.ylabel("akurasi")
plt.title("Akurasi per sesi")
plt.ylim(0, 1)
plt.show()

### ✍️ TULIS DI SINI
Sesi mana yang paling sulit untuk semua model? Model mana yang paling stabil?

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*

---
## Langkah 6 — Kesimpulan
💾 Pesan commit contoh: `tulis kesimpulan eksperimen ensemble`

In [ ]:
display(pd.DataFrame(hasil).T.sort_values("macro_f1", ascending=False))

### ✍️ TULIS DI SINI
Tulis 3–4 kalimat: model ensemble mana yang terbaik, apakah hasilnya mengubah kesimpulan di slide 6, dan apa yang kamu pelajari.

*(klik dua kali sel ini, hapus teks petunjuk, lalu tulis jawabanmu)*